In [ ]:
from PIL import Image
from dataclasses import dataclass

@dataclass
class ImageTile:
    image: Image.Image
    bbox: tuple[int, int, int, int]  # (x0, y0, x1, y1) in original image coords

_GRID_LAYOUTS = {
    2: (1, 2),  # rows, cols
    4: (2, 2),
    8: (2, 4),
}

def split_image(image: Image.Image, split: int, overlap: float = 0.0) -> list[ImageTile]:
    """
    Split an image into `split` tiles (2, 4, or 8) arranged in a grid,
    with optional fractional overlap between adjacent tiles.

    Args:
        image: source PIL image
        split: number of tiles — must be 2, 4, or 8
        overlap: fraction of tile width/height to overlap with neighbors (0.0–0.4)

    Returns:
        List of ImageTile, each with the crop and its bbox in original coords.
    """
    if split not in _GRID_LAYOUTS:
        raise ValueError(f"split must be one of {list(_GRID_LAYOUTS)}, got {split}")
    if not (0.0 <= overlap < 0.5):
        raise ValueError("overlap must be in [0, 0.5)")

    rows, cols = _GRID_LAYOUTS[split]
    W, H = image.size
    base_tile_w = W / cols
    base_tile_h = H / rows

    tiles = []
    for r in range(rows):
        for c in range(cols):
            # nominal (non-overlapping) tile bounds
            x0 = c * base_tile_w
            y0 = r * base_tile_h
            x1 = x0 + base_tile_w
            y1 = y0 + base_tile_h

            # expand by overlap, clamped to image bounds
            ox = base_tile_w * overlap
            oy = base_tile_h * overlap
            ex0 = max(0, x0 - ox)
            ey0 = max(0, y0 - oy)
            ex1 = min(W, x1 + ox)
            ey1 = min(H, y1 + oy)

            bbox = (int(ex0), int(ey0), int(ex1), int(ey1))
            crop = image.crop(bbox)
            tiles.append(ImageTile(image=crop, bbox=bbox))

    return tiles

In [7]:
{
  "name": "split_image",
  "description": "Split the current image into a grid of sub-images for closer inspection. Use when text, tags, or symbols are too small or unclear to read at full resolution.",
  "input_schema": {
    "type": "object",
    "properties": {
      "split": {
        "type": "integer",
        "enum": [2, 4, 8],
        "description": "Number of tiles to split into: 2 (side-by-side), 4 (2x2 grid), 8 (2x4 grid)"
      }
    },
    "required": ["split"]
  }
}

{'name': 'split_image',
 'description': 'Split the current image into a grid of sub-images for closer inspection. Use when text, tags, or symbols are too small or unclear to read at full resolution.',
 'input_schema': {'type': 'object',
  'properties': {'split': {'type': 'integer',
    'enum': [2, 4, 8],
    'description': 'Number of tiles to split into: 2 (side-by-side), 4 (2x2 grid), 8 (2x4 grid)'}},
  'required': ['split']}}

In [23]:
from PIL import Image
from dataclasses import dataclass

@dataclass
class ImageTile:
    image: Image.Image
    bbox: tuple[int, int, int, int]  # (x0, y0, x1, y1) in original image coords

_GRID_LAYOUTS = {
    2: (1, 2),  # rows, cols
    4: (2, 2),
    8: (2, 4),
}

def split_image(image: Image.Image, split: int, overlap: float = 0.0) -> list[ImageTile]:
    if split not in _GRID_LAYOUTS:
        raise ValueError(f"split must be one of {list(_GRID_LAYOUTS)}, got {split}")
    if not (0.0 <= overlap < 0.5):
        raise ValueError("overlap must be in [0, 0.5)")

    rows, cols = _GRID_LAYOUTS[split]
    W, H = image.size
    base_tile_w = W / cols
    base_tile_h = H / rows

    tiles = []
    for r in range(rows):
        for c in range(cols):
            x0 = c * base_tile_w
            y0 = r * base_tile_h
            x1 = x0 + base_tile_w
            y1 = y0 + base_tile_h

            ox = base_tile_w * overlap
            oy = base_tile_h * overlap
            ex0 = max(0, x0 - ox)
            ey0 = max(0, y0 - oy)
            ex1 = min(W, x1 + ox)
            ey1 = min(H, y1 + oy)

            bbox = (int(ex0), int(ey0), int(ex1), int(ey1))
            crop = image.crop(bbox)
            tiles.append(ImageTile(image=crop, bbox=bbox))

    return tiles


def split_tile(tile: ImageTile, split: int, overlap: float = 0.0) -> list[ImageTile]:
    sub_tiles = split_image(tile.image, split, overlap)
    parent_x0, parent_y0, _, _ = tile.bbox
    composed = []
    for st in sub_tiles:
        sx0, sy0, sx1, sy1 = st.bbox
        global_bbox = (parent_x0 + sx0, parent_y0 + sy0, parent_x0 + sx1, parent_y0 + sy1)
        composed.append(ImageTile(image=st.image, bbox=global_bbox))
    return composed

In [24]:
from PIL import Image

# load your P&ID
img = Image.open("123_page-0001.jpg")
print("original size:", img.size)

tiles = split_image(img, split=4, overlap=0.1)
for i, t in enumerate(tiles):
    print(f"tile {i}: bbox={t.bbox}, size={t.image.size}")
    t.image.save(f"tile_{i}.png")  # save so you can eyeball each crop

# level 2: split tile[0] again, since bboxes compose correctly
sub_tiles = split_tile(tiles[0], split=4, overlap=0.1)
for i, st in enumerate(sub_tiles):
    print(f"tile 0 -> sub {i}: bbox={st.bbox} (should be relative to ORIGINAL image, not tile 0)")
    st.image.save(f"tile_0_sub_{i}.png")

original size: (3509, 2480)
tile 0: bbox=(0, 0, 1929, 1364), size=(1929, 1364)
tile 1: bbox=(1579, 0, 3509, 1364), size=(1930, 1364)
tile 2: bbox=(0, 1116, 1929, 2480), size=(1929, 1364)
tile 3: bbox=(1579, 1116, 3509, 2480), size=(1930, 1364)
tile 0 -> sub 0: bbox=(0, 0, 1060, 750) (should be relative to ORIGINAL image, not tile 0)
tile 0 -> sub 1: bbox=(868, 0, 1929, 750) (should be relative to ORIGINAL image, not tile 0)
tile 0 -> sub 2: bbox=(0, 613, 1060, 1364) (should be relative to ORIGINAL image, not tile 0)
tile 0 -> sub 3: bbox=(868, 613, 1929, 1364) (should be relative to ORIGINAL image, not tile 0)


In [10]:
pip install httpx pillow

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached certifi-2026.6.17-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
Using cached idna-3.18-py3-none-any.whl (65 kB)
Using cached certifi-2026.6.17-py3-none-any.whl (133 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [httpx]32m4/6 [anyio]
Note: you may need to restart the kernel to use updated packages.


In [14]:
!pip install python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)


In [25]:
import asyncio
import base64
import io
import json
import httpx
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("AGNES_API_KEY")
BASE_URL = "https://apihub.agnes-ai.com/v1/chat/completions"
MODEL = "agnes-2.0-flash"

MAX_CONCURRENT = 5
MAX_DEPTH = 3
MIN_TILE_PIXELS = 150

sem = asyncio.Semaphore(MAX_CONCURRENT)

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "split_further",
            "description": "Split this image into smaller tiles because text/symbols are unclear.",
            "parameters": {
                "type": "object",
                "properties": {
                    "split": {
                        "type": "integer",
                        "enum": [2, 4, 8]
                    }
                },
                "required": ["split"]
            }
        }
    },
    {
    "type": "function",
    "function": {
        "name": "report_extraction",
        "description": "Extract everything identifiable from this P&ID crop.",
        "parameters": {
            "type": "object",
            "properties": {
                "items": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "tag": {
                                "type": "string",
                                "description": "Equipment or instrument tag, e.g. P-101"
                            },
                            "type": {
                                "type": "string",
                                "description": "Equipment type, e.g. Pump, Valve, Vessel, Instrument"
                            },
                            "description": {
                                "type": "string",
                                "description": "Any readable description or annotation"
                            },
                            "line_number": {
                                "type": "string",
                                "description": "Pipeline number if visible"
                            },
                            "service": {
                                "type": "string",
                                "description": "Fluid or service if mentioned"
                            }
                        },
                        "required": ["tag", "type"]
                    }
                }
            },
            "required": ["items"]
        }
    }
}
]


def image_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()


async def call_model(image_b64, tools):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Extract all equipment tags, instruments, and line numbers "
                            "visible in this P&ID crop. "
                            "If anything is too small or unclear to read confidently, "
                            "call split_further instead of guessing."
                        )
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{image_b64}"
                        }
                    }
                ]
            }
        ],
        "tools": tools,
        "tool_choice": "auto"
    }

    async with httpx.AsyncClient(timeout=120) as client:
        response = await client.post(
            BASE_URL,
            headers=headers,
            json=payload,
        )
        response.raise_for_status()
        return response.json()


async def process_tile(tile: ImageTile, depth: int):

    w, h = tile.image.size

    if w < MIN_TILE_PIXELS or h < MIN_TILE_PIXELS or depth >= MAX_DEPTH:
        tools = [TOOLS[1]]
    else:
        tools = TOOLS

    async with sem:
        response = await call_model(image_to_b64(tile.image), tools)

    message = response["choices"][0]["message"]

    tool_calls = message.get("tool_calls", [])

    if not tool_calls:
        return []

    tool_call = tool_calls[0]

    function_name = tool_call["function"]["name"]
    arguments = json.loads(tool_call["function"]["arguments"])

    if function_name == "report_extraction":

        results = []

        for item in arguments["items"]:

            local = item.get("local_bbox")
            global_bbox = None

            if local:
                x0, y0, x1, y1 = local
                px0, py0, _, _ = tile.bbox
                global_bbox = (
                    px0 + x0,
                    py0 + y0,
                    px0 + x1,
                    py0 + y1,
                )

            results.append(
                {
                    **item,
                    "global_bbox": global_bbox,
                    "source_bbox": tile.bbox,
                    "depth": depth,
                }
            )

        return results

    elif function_name == "split_further":

        split_n = arguments["split"]

        sub_tiles = split_tile(
            tile,
            split=split_n,
            overlap=0.1,
        )

        sub_results = await asyncio.gather(
            *[
                process_tile(st, depth + 1)
                for st in sub_tiles
            ]
        )

        return [
            item
            for sub in sub_results
            for item in sub
        ]

    return []


async def run_on_image(image, initial_split=4):


    top_tiles = split_image(
        image,
        split=initial_split,
        overlap=0.1,
    )

    all_results = await asyncio.gather(
        *[
            process_tile(tile, 0)
            for tile in top_tiles
        ]
    )

    return [
        item
        for sub in all_results
        for item in sub
    ]

In [26]:
# image = Image.open("123_page-0001.jpg")
image = Image.open("tile_2.png")


In [27]:
# results = await run_on_image(image)

In [28]:
# results

In [29]:
response = await call_model(
    image_to_b64(image),
    [TOOLS[1]],
)

import json
print(json.dumps(response, indent=2))

{
  "id": "chatcmpl-a5ee69797dc73eee",
  "created": 1784297457,
  "model": "agnes-2.0-flash",
  "object": "chat.completion",
  "system_fingerprint": "vllm-0.21.0-tp2-84122c74",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "message": {
        "role": "assistant",
        "tool_calls": [
          {
            "function": {
              "arguments": "{\"items\": [{\"tag\": \"P-01\", \"type\": \"Pump\", \"description\": \"Lube Oil Pump\"}, {\"tag\": \"P-02\", \"type\": \"Pump\", \"description\": \"Lube Oil Pump\"}, {\"tag\": \"PG-001\", \"type\": \"Instrument\", \"description\": \"Pressure Gauge\"}, {\"tag\": \"PG-005\", \"type\": \"Instrument\", \"description\": \"Pressure Gauge\"}, {\"tag\": \"PZV-001\", \"type\": \"Valve\", \"description\": \"Pressure Safety Valve\"}, {\"tag\": \"PZV-002\", \"type\": \"Valve\", \"description\": \"Pressure Safety Valve\"}, {\"tag\": \"TG-003\", \"type\": \"Instrument\", \"description\": \"Temperature Gauge\"}, {\"

In [21]:
response = await call_model(
    image_to_b64(image),
    [TOOLS[1]],
)

import json
print(json.dumps(response, indent=2))

{
  "id": "chatcmpl-9177707dd70e401f",
  "created": 1784297045,
  "model": "agnes-2.0-flash",
  "object": "chat.completion",
  "system_fingerprint": "vllm-0.21.0-tp2-84122c74",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "message": {
        "role": "assistant",
        "tool_calls": [
          {
            "function": {
              "arguments": "{\"items\": [{\"tag\": \"P-01\", \"type\": \"equipment\", \"local_bbox\": [193, 786, 235, 824]}, {\"tag\": \"P-02\", \"type\": \"equipment\", \"local_bbox\": [296, 786, 335, 824]}, {\"tag\": \"Air Cooler\", \"type\": \"equipment\", \"local_bbox\": [327, 574, 425, 601]}, {\"tag\": \"OIL FILTER\", \"type\": \"equipment\", \"local_bbox\": [558, 624, 607, 657]}, {\"tag\": \"OIL FILTER\", \"type\": \"equipment\", \"local_bbox\": [558, 716, 607, 749]}, {\"tag\": \"Lube and Seal Oil Tank\", \"type\": \"equipment\", \"local_bbox\": [546, 869, 639, 880]}, {\"tag\": \"Compressor K-01\", \"type\": \"equipment\", 

In [31]:
import asyncio
import base64
import io
import json
import os

import httpx
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("AGNES_API_KEY")
BASE_URL = "https://apihub.agnes-ai.com/v1/chat/completions"
MODEL = "agnes-2.0-flash"

MAX_DEPTH = 3
MIN_TILE_PIXELS = 150

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "split_further",
            "description": "Split this image into smaller tiles because text or symbols are too small to read.",
            "parameters": {
                "type": "object",
                "properties": {
                    "split": {
                        "type": "integer",
                        "enum": [2, 4, 8]
                    }
                },
                "required": ["split"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "report_extraction",
            "description": "Extract everything identifiable from this P&ID crop.",
            "parameters": {
                "type": "object",
                "properties": {
                    "items": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "tag": {
                                    "type": "string"
                                },
                                "type": {
                                    "type": "string"
                                },
                                "description": {
                                    "type": "string"
                                },
                                "line_number": {
                                    "type": "string"
                                },
                                "service": {
                                    "type": "string"
                                }
                            },
                            "required": ["tag", "type"]
                        }
                    }
                },
                "required": ["items"]
            }
        }
    }
]


def image_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()


async def call_model(image_b64, tools):

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Extract every identifiable piece of information from this "
                            "P&ID crop.\n\n"
                            "Return:\n"
                            "- equipment tag\n"
                            "- equipment type\n"
                            "- line number\n"
                            "- description\n"
                            "- service/fluid\n\n"
                            "If any text or symbols are too small to read confidently, "
                            "call split_further instead of guessing."
                        )
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{image_b64}"
                        }
                    }
                ]
            }
        ],
        "tools": tools,
        "tool_choice": "auto"
    }

    # Retry with exponential backoff
    for attempt in range(5):

        async with httpx.AsyncClient(timeout=120) as client:
            response = await client.post(
                BASE_URL,
                headers=headers,
                json=payload,
            )

        if response.status_code == 429:
            wait = 2 ** attempt
            print(f"⚠️ Rate limited. Waiting {wait} seconds...")
            await asyncio.sleep(wait)
            continue

        response.raise_for_status()

        # Small delay even after successful requests
        await asyncio.sleep(1)

        return response.json()

    raise Exception("Too many retries due to rate limiting.")


async def process_tile(tile: ImageTile, depth: int):

    print(f"Processing depth={depth}, tile={tile.bbox}")

    w, h = tile.image.size

    if w < MIN_TILE_PIXELS or h < MIN_TILE_PIXELS or depth >= MAX_DEPTH:
        tools = [TOOLS[1]]
    else:
        tools = TOOLS

    response = await call_model(image_to_b64(tile.image), tools)

    message = response["choices"][0]["message"]

    tool_calls = message.get("tool_calls", [])

    if not tool_calls:
        return []

    tool_call = tool_calls[0]

    function_name = tool_call["function"]["name"]
    arguments = json.loads(tool_call["function"]["arguments"])

    # ----------------------------------------------------
    # Extraction complete
    # ----------------------------------------------------

    if function_name == "report_extraction":

        return arguments["items"]

    # ----------------------------------------------------
    # Split further
    # ----------------------------------------------------

    elif function_name == "split_further":

        split_n = arguments["split"]

        print(f"Splitting tile {tile.bbox} into {split_n}x{split_n}")

        sub_tiles = split_tile(
            tile,
            split=split_n,
            overlap=0.1,
        )

        results = []

        # Sequential recursion
        for st in sub_tiles:

            sub_result = await process_tile(
                st,
                depth + 1,
            )

            results.extend(sub_result)

        return results

    return []


async def run_on_image(image, initial_split=4):

    top_tiles = split_image(
        image,
        split=initial_split,
        overlap=0.1,
    )

    results = []

    print(f"Total top-level tiles: {len(top_tiles)}")

    # Sequential processing
    for i, tile in enumerate(top_tiles):

        print(f"\nTile {i+1}/{len(top_tiles)}")

        tile_results = await process_tile(
            tile,
            depth=0,
        )

        results.extend(tile_results)

    return results

In [32]:
from PIL import Image

image = Image.open("123_page-0001.jpg")

results = await run_on_image(image, initial_split=4)

results

Total top-level tiles: 4

Tile 1/4
Processing depth=0, tile=(0, 0, 1929, 1364)

Tile 2/4
Processing depth=0, tile=(1579, 0, 3509, 1364)

Tile 3/4
Processing depth=0, tile=(0, 1116, 1929, 2480)
Splitting tile (0, 1116, 1929, 2480) into 4x4
Processing depth=1, tile=(0, 1116, 1060, 1866)
Processing depth=1, tile=(868, 1116, 1929, 1866)
Splitting tile (868, 1116, 1929, 1866) into 4x4
Processing depth=2, tile=(868, 1116, 1451, 1528)
Processing depth=2, tile=(1345, 1116, 1929, 1528)
Splitting tile (1345, 1116, 1929, 1528) into 4x4
Processing depth=3, tile=(1345, 1116, 1666, 1342)
Processing depth=3, tile=(1607, 1116, 1929, 1342)
Processing depth=3, tile=(1345, 1301, 1666, 1528)
Processing depth=3, tile=(1607, 1301, 1929, 1528)
Processing depth=2, tile=(868, 1453, 1451, 1866)
Processing depth=2, tile=(1345, 1453, 1929, 1866)
Processing depth=1, tile=(0, 1729, 1060, 2480)
Processing depth=1, tile=(868, 1729, 1929, 2480)
Splitting tile (868, 1729, 1929, 2480) into 4x4
Processing depth=2, tile=(

ReadTimeout: 

In [33]:
results

NameError: name 'results' is not defined

Top tile 1/4
Depth=0 bbox=(0, 0, 1929, 1364)


KeyError: 'tool_calls'